# jaxsedfit: DR16Q quasars with BandWagon

A bulk workflow for catalog SED fitting:

1. download the SDSS DR16Q quasar property catalog (Wu & Shen 2022) and draw a random sample;
2. cross-match GALEX, SDSS and AllWISE photometry with [BandWagon](https://github.com/burke86/bandwagon) through CDS XMatch;
3. fit each SED with a fast MAP fit (`method="optax"`).

The catalog is a 1.4 GB download, cached by astropy after the first run. `SAMPLE_SIZE` is small to keep the demo quick. You need internet access, `bandwagon`, and the DSPS SSP file `tempdata.h5` in the repository root.

In [ ]:
import numpy as np
import numpyro.distributions as dist
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.utils.data import download_file
from bandwagon import matches_to_photometry, xmatch_catalogs

from jaxsedfit import (
    AGNConfig,
    FilterSet,
    FitConfig,
    GalaxyConfig,
    InferenceConfig,
    JAXSEDFit,
    LikelihoodConfig,
    Observation,
    PhotometryData,
    PriorConfig,
    load_filter_curves,
)

## 1. Draw a sample from DR16Q

In [ ]:
DR16Q_URL = "http://quasar.astro.illinois.edu/paper_data/DR16Q/dr16q_prop_May01_2024.fits.gz"
SAMPLE_SIZE = 3
MIN_BANDS = 5

dr16q = Table.read(download_file(DR16Q_URL, cache=True), hdu=1)
dr16q = dr16q[np.isfinite(dr16q["Z_DR16Q"]) & (dr16q["Z_DR16Q"] > 0)]

rng = np.random.default_rng(20260419)
sample = dr16q[rng.choice(len(dr16q), size=SAMPLE_SIZE, replace=False)]
source_ids = [f"DR16Q_{name.strip()}" for name in sample["SDSS_NAME"]]
redshifts = dict(zip(source_ids, sample["Z_DR16Q"]))
sample["SDSS_NAME", "RA", "DEC", "Z_DR16Q"]

## 2. Cross-match photometry

`xmatch_catalogs` queries GALEX, SDSS DR16 and AllWISE (three CDS XMatch requests), and `matches_to_photometry` converts the magnitudes into a long-form table of fluxes in mJy with nominal PSF sizes.

In [ ]:
coords = SkyCoord(sample["RA"], sample["DEC"], unit="deg")
matches = xmatch_catalogs(coords, source_id=source_ids)
photometry = matches_to_photometry(matches, max_mag_err=1.0)
photometry["source_id", "filter_name", "flux_mjy", "flux_err_mjy", "psf_fwhm_arcsec"]

## 3. Fit each quasar

A 3% error floor keeps the heterogeneous, multi-epoch photometry from being over-weighted. `variability_uncertainty=True` adds the expected scatter from quasar variability between epochs.

In [ ]:
def make_config(source_id, phot):
    flux = np.asarray(phot["flux_mjy"], dtype=float)
    err = np.maximum(np.asarray(phot["flux_err_mjy"], dtype=float), 0.03 * flux)
    return FitConfig(
        observation=Observation(object_id=source_id, redshift=float(redshifts[source_id])),
        photometry=PhotometryData(
            filter_names=list(phot["filter_name"]),
            fluxes=list(flux),
            errors=list(err),
            psf_fwhm_arcsec=list(phot["psf_fwhm_arcsec"]),
        ),
        filters=FilterSet(curves=load_filter_curves(list(phot["filter_name"]))),
        galaxy=GalaxyConfig(dsps_ssp_fn="../tempdata.h5"),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(systematics_width=0.08, variability_uncertainty=True, use_host_capture_model=True),
        inference=InferenceConfig(method="optax", map_steps=600, learning_rate=5e-3),
        prior_config=PriorConfig(
            stellar_mass=dist.Normal(10.5, 1.25),
            host={"ebv_gal": dist.HalfNormal(0.2)},
            agn={"ebv_agn": dist.HalfNormal(0.2)},
        ),
    )


fitters = {}
for source_id in source_ids:
    phot = photometry[photometry["source_id"] == source_id]
    if len(phot) < MIN_BANDS:
        print(f"{source_id}: only {len(phot)} bands, skipped")
        continue
    fitters[source_id] = JAXSEDFit(make_config(source_id, phot))
    fitters[source_id].fit(progress_bar=False)

Table(
    rows=[
        (sid, redshifts[sid], f.summary()["log_stellar_mass_median"], f.summary()["log_agn_bol_luminosity_median"])
        for sid, f in fitters.items()
    ],
    names=("source_id", "redshift", "log_stellar_mass", "log_agn_bol_luminosity"),
)

## 4. Inspect the fits

In [ ]:
for source_id, fitter in fitters.items():
    fitter.plot_sed(show=True, title=source_id)

## Notes

- GALEX and SDSS magnitudes are AB; BandWagon converts AllWISE Vega magnitudes with the standard WISE zero points.
- The raw XMatch tables are in `matches` if you need catalog quality flags.
- Save intermediate tables with `photometry.write("photometry.ecsv")` to avoid repeating the cross-match.